<h1>Chapter 2 - Tokens and Token Embeddings</h1>
<i>Exploring tokens and embeddings as an integral part of building LLMs</i>


<a href="https://www.amazon.com/Hands-Large-Language-Models-Understanding/dp/1098150961"><img src="https://img.shields.io/badge/Buy%20the%20Book!-grey?logo=amazon"></a>
<a href="https://www.oreilly.com/library/view/hands-on-large-language/9781098150952/"><img src="https://img.shields.io/badge/O'Reilly-white.svg?logo=data:image/svg%2bxml;base64,PHN2ZyB3aWR0aD0iMzQiIGhlaWdodD0iMjciIHZpZXdCb3g9IjAgMCAzNCAyNyIgZmlsbD0ibm9uZSIgeG1sbnM9Imh0dHA6Ly93d3cudzMub3JnLzIwMDAvc3ZnIj4KPGNpcmNsZSBjeD0iMTMiIGN5PSIxNCIgcj0iMTEiIHN0cm9rZT0iI0Q0MDEwMSIgc3Ryb2tlLXdpZHRoPSI0Ii8+CjxjaXJjbGUgY3g9IjMwLjUiIGN5PSIzLjUiIHI9IjMuNSIgZmlsbD0iI0Q0MDEwMSIvPgo8L3N2Zz4K"></a>
<a href="https://github.com/HandsOnLLM/Hands-On-Large-Language-Models"><img src="https://img.shields.io/badge/GitHub%20Repository-black?logo=github"></a>
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/HandsOnLLM/Hands-On-Large-Language-Models/blob/main/chapter02/Chapter%202%20-%20Tokens%20and%20Token%20Embeddings.ipynb)

---

This notebook is for Chapter 2 of the [Hands-On Large Language Models](https://www.amazon.com/Hands-Large-Language-Models-Understanding/dp/1098150961) book by [Jay Alammar](https://www.linkedin.com/in/jalammar) and [Maarten Grootendorst](https://www.linkedin.com/in/mgrootendorst/).

---

<a href="https://www.amazon.com/Hands-Large-Language-Models-Understanding/dp/1098150961">
<img src="https://raw.githubusercontent.com/HandsOnLLM/Hands-On-Large-Language-Models/main/images/book_cover.png" width="350"/></a>


## Ligação com a aula "Tokens e Embeddings"

Este notebook é a prática que acompanha a aula "Tokens e Embeddings". Vale relembrar o caminho percorrido na aula antes de executar o codigo:

1. **One-Hot Encoding**: cada palavra vira um vetor esparso e ortogonal. Problema: dimensionalidade explode com o vocabulario e nenhuma relacao semantica e capturada (produto interno sempre zero).
2. **Bag of Words e TF-IDF**: pesos estatisticos ajudam na recuperação de documentos, mas ainda não capturam sinonimia. Frases com o mesmo sentido ("O felino bebeu leite" vs "O gato tomou laticinio") continuam com similaridade zero.
3. **Embeddings**: solução matematica baseada na hipótese distribucional de Firth (1957), "uma palavra e conhecida pela companhia que mantém". Palavras são imersas em um espaco vetorial denso e contínuo (dezenas a poucas centenas de dimensões), onde direção e proximidade passam a codificar significado.

Neste notebook vamos observar esses conceitos na prática, em duas frentes:

- **Tokens**: como um LLM real (Phi-3-mini) fatia um prompt em tokens antes de processá-lo, e como   diferentes tokenizadores (BPE, WordPiece) decompoem o mesmo texto de formas distintas.
- **Embeddings**: como tokens viram vetores (embeddings estáticos vs. contextualizados), como frases inteiras podem virar um unico vetor, e como a mesma lógica de Word2Vec usada para palavras pode ser reaproveitada para recomendar músicas a partir de playlists.

Fonte: Alammar, J.; Grootendorst, M. *Hands-On Large Language Models*, Capitulo 2. O'Reilly, 2024.


Esta célula instala o **gensim**, biblioteca que usaremos em duas partes mais adiante: (i) para carregar *embeddings* de palavras pré-treinados no estilo GloVe/Word2Vec (secao "Word Embeddings Beyond LLMs") e (ii) para treinar nosso proprio modelo Word2Vec sobre playlists de músicas (seção final de recomendação). Note que essas duas partes não usam LLM nenhum: são *embeddings* "classicos", da geração anterior aos Transformers, mas que seguem exatamente a mesma ideia discutida na aula.


In [1]:
!pip install gensim

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.8/27.8 MB 47.8 MB/s eta 0:00:00


# Baixando e executando um LLM

Vamos carregar o modelo **microsoft/Phi-3-mini-4k-instruct** e seu tokenizador. Repare que são dois objetos carregados separadamente: o `tokenizer` (que so fatia texto em tokens e converte para IDs) e o `model` (a rede neural que efetivamente processa esses IDs). Essa separação e a mesma mostrada no slide
"Um modelo de linguagem armazena embeddings para o vocabulário de seu tokenizador": o tokenizador e treinado uma vez e fica atrelado ao modelo, que armazena uma tabela de embeddings (uma linha por token do vocabulário). Por isso um modelo pré-treinado não pode simplesmente trocar de tokenizador sem
retreinar.


In [35]:
from transformers import AutoModelForCausalLM, AutoTokenizer

# Load model and tokenizer
model = AutoModelForCausalLM.from_pretrained(
    "microsoft/Phi-3-mini-4k-instruct",
    device_map="cuda",
    torch_dtype="auto",
    trust_remote_code=False,
)
tokenizer = AutoTokenizer.from_pretrained("microsoft/Phi-3-mini-4k-instruct")

Loading weights:   0%|          | 0/195 [00:00<?, ?it/s]

Aqui reproduzimos o fluxo completo visto na aula (slide "Como os tokenizadores preparam as entradas para o modelo de linguagem"): o prompt em texto passa pelo `tokenizer` (etapa *encode*), vira uma sequência de IDs de token, e so entao e enviado ao `model.generate`. O modelo nao produz a resposta inteira de uma vez: ele gera **um token por vez**, cada novo token dependendo dos anteriores (o mesmo comportamento mostrado no exemplo do terminal com `ollama` na aula). Ao final, `tokenizer.decode` converte os IDs gerados de volta em texto legível (etapa *decode*).


In [36]:
prompt = "Write an email apologizing to Sarah for the tragic gardening mishap. Explain how it happened.<|assistant|>"

# Tokenize the input prompt
input_ids = tokenizer(prompt, return_tensors="pt").input_ids.to("cuda")

# Generate the text
generation_output = model.generate(
  input_ids=input_ids,
  max_new_tokens=20
)

# Print the output
print(tokenizer.decode(generation_output[0]))

Write an email apologizing to Sarah for the tragic gardening mishap. Explain how it happened.<|assistant|> Subject: Heartfelt Apologies for the Gardening Mishap


Dear Sarah,


I hope this message finds you well. I am writing to express my deepest apologies for the unfortunate incident that


> <|assistant|> token para indicar que a LLM deve responder logo em seguida.

`input_ids` e o resultado da etapa de *encode*: não e o texto, e sim a sequência numérica de IDs de token que o modelo efetivamente recebe como entrada. Cada número corresponde a uma linha da tabela de vocabulário do tokenizador.


In [4]:
print(input_ids)

tensor([[14350,   385,  4876, 27746,  5281,   304, 19235,   363,   278, 25305,
           293, 16423,   292,   286,   728,   481, 29889, 12027,  7420,   920,
           372,  9559, 29889, 32001]], device='cuda:0')


Aqui decodificamos cada ID individualmente para ver exatamente como o prompt foi fatiado. Este e o mesmo exercício do slide "Como os tokenizadores preparam as entradas para o modelo de linguagem", onde a frase "Have the bards who preceded..." era quebrada em tokens como `Have`, `the`, `b`, `ards`, `who`
(note que "bards" não existe como token único e precisa ser reconstruída a partir de `b` + `ards`, um exemplo de tokenizacao por subpalavra).


In [5]:
for id in input_ids[0]:
   print(tokenizer.decode(id))

Write
an
email
apolog
izing
to
Sarah
for
the
trag
ic
garden
ing
m
ish
ap
.
Exp
lain
how
it
happened
.
<|assistant|>


`generation_output` e o tensor bruto retornado pelo modelo: a concatenação dos IDs do prompt de entrada com os novos IDs gerados pelo modelo. Ainda está em formato numérico, por isso o `tokenizer.decode` foi necessário na célula anterior para exibir o texto final.


In [6]:
generation_output

tensor([[14350,   385,  4876, 27746,  5281,   304, 19235,   363,   278, 25305,
           293, 16423,   292,   286,   728,   481, 29889, 12027,  7420,   920,
           372,  9559, 29889, 32001,  3323,   622, 29901, 17778, 29888,  2152,
          6225, 11763,   363,   278, 19906,   292,   341,   728,   481,    13,
            13,    13, 29928,   799]], device='cuda:0')

Este é exatamente o exemplo apresentado na aula (slide "Como os tokenizadores preparam as entradas para o modelo de linguagem", com a captura de tela mostrando `tokenizer.decode(3323)` etc.): a palavra "Subject" foi tokenizada em duas subpalavras, `Sub` e `ject`, que juntas reconstroem a palavra completa.
Isso ilustra na prática o que os slides "Tokens de Subpalavras" descreveram: o tokenizador usa um token para a raiz/préfixo e outro para o sufixo, permitindo representar palavras raras ou compostas sem precisar de um token exclusivo para cada uma delas (evitando o problema do vocabulario `<UNK>` da tokenização por palavra inteira).


In [7]:
print(tokenizer.decode(3323))
print(tokenizer.decode(622))
print(tokenizer.decode([3323, 622]))
print(tokenizer.decode(29901))

Sub
ject
Subject
:


# Comparando tokenizadores de diferentes LLMs

Na aula vimos que o comportamento de um tokenizador e determinado por três fatores (slide "Propriedades do Tokenizer"): o **método de tokenizacao** escolhido (BPE, usado pelos modelos GPT, ou WordPiece, usado pelo BERT), os **parâmetros do tokenizador** (tamanho do vocabulário, tokens especiais) e o
**domínio dos dados** em que ele foi treinado. Nesta seção vamos comparar, lado a lado, como tokenizadores de modelos diferentes (BERT, GPT-2, T5, GPT-4, StarCoder2, Galactica, Phi-3) fatiam o mesmo texto contendo casos dificeis: maiúsculas, emojis, caracteres em chines, palavras reservadas de código, tabulações e uma expressão aritmética.


Esta função auxiliar `show_tokens` carrega o tokenizador indicado, tokeniza a frase e imprime cada token com uma cor de fundo diferente, facilitando visualizar onde o tokenizador decidiu "cortar" o texto (o mesmo tipo de visualização colorida usada nos slides de embeddings de palavra, só que aqui
aplicada aos tokens em vez das dimensões do vetor).


In [8]:
from transformers import AutoModelForCausalLM, AutoTokenizer

colors_list = [
    '102;194;165', '252;141;98', '141;160;203',
    '231;138;195', '166;216;84', '255;217;47'
]

def show_tokens(sentence, tokenizer_name):
    tokenizer = AutoTokenizer.from_pretrained(tokenizer_name)
    token_ids = tokenizer(sentence).input_ids
    for idx, t in enumerate(token_ids):
        print(
            f'\x1b[0;30;48;2;{colors_list[idx % len(colors_list)]}m' +
            tokenizer.decode(t) +
            '\x1b[0m',
            end=' '
        )

O texto de teste foi montado propositalmente com casos que expõem diferenças entre tokenizadores: letras maiúsculas ("CAPITALIZATION"), emoji e caractere em chinês (testando cobertura Unicode via tokens de byte, como no slide "Tokens de Byte"), palavras-chave de programação, espaços e tabulações, e uma expressão numérica. Isso e análogo ao caso "Biomimetica" discutido na aula: veremos se cada tokenizador reconhece essas unidades como token único ou precisa fragmentá-las em pedaços menores.


In [9]:
text = """
English and CAPITALIZATION
🎵 鸟
show_tokens False None elif == >= else: two tabs:"    " Three tabs: "       "
12.0*50=600
"""

**BERT (uncased).** Usa WordPiece. Por ser *uncased*, o tokenizador primeiro converte tudo para minúsculas antes de tokenizar, então a informação de "CAPITALIZATION" é perdida.


In [10]:
show_tokens(text, "bert-base-uncased")

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

[CLS] english and capital ##ization [UNK] [UNK] show _ token ##s false none eli ##f = = > = else : two tab ##s : " " three tab ##s : " " 12 . 0 * 50 = 600 [SEP] 

> [CLS], [UNK] https://h2o.ai/wiki/classify-token/

**BERT (cased).** Mesma familia WordPiece, mas preserva maiúsculas e minúsculas como tokens distintos, o que aumenta o vocabulário mas mantem informação que o uncased descarta.


In [11]:
show_tokens(text, "bert-base-cased")

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/213k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/436k [00:00<?, ?B/s]

[CLS] English and CA ##PI ##TA ##L ##I ##Z ##AT ##ION [UNK] [UNK] show _ token ##s F ##als ##e None el ##if = = > = else : two ta ##bs : " " Three ta ##bs : " " 12 . 0 * 50 = 600 [SEP] 

**GPT-2.** Usa Byte Pair Encoding (BPE), o método citado no slide "Como o tokenizador decompõe o texto?" como o mais utilizado pelos modelos da familia GPT. Repare como espaços costumam ser incorporados ao inicio do token seguinte, em vez de virarem um token separado.


In [12]:
show_tokens(text, "gpt2")

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]


 English  and  CAP ITAL IZ ATION 
 � � �  � � � 
 show _ t ok ens  False  None  el if  ==  >=  else :  two  tabs :"        "  Three  tabs :  "              " 
 12 . 0 * 50 = 600 
 

**Flan-T5 (small).** Tokenizador diferente (SentencePiece/Unigram), treinado em um corpus e domínio distintos dos anteriores, o que muda a granularidade da tokenizacao mesmo para o mesmo texto de entrada.


In [13]:
show_tokens(text, "google/flan-t5-small")

config.json:   0%|          | 0.00/1.40k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/2.54k [00:00<?, ?B/s]

spiece.model: reconstructing file:   0%|          |  0.00B /  792kB            

spiece.model: downloading bytes:           |  0.00B            

tokenizer.json:   0%|          | 0.00/2.42M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/2.20k [00:00<?, ?B/s]

English and CA PI TAL IZ ATION  <unk>  <unk> show _ to ken s Fal s e None  e l if = = > = else : two tab s : " " Three tab s : " " 12. 0 * 50 = 600 </s> 

**GPT-4 (via Xenova, equivalente ao `tiktoken`).** Vocabulário bem maior que o do GPT-2, o que tende a produzir tokens mais longos (menos fragmentação) para o mesmo texto.


In [14]:
# The official is `tiktoken` but this the same tokenizer on the HF platform
show_tokens(text, "Xenova/gpt-4")

tokenizer_config.json:   0%|          | 0.00/460 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.01M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/917k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/98.0 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/4.23M [00:00<?, ?B/s]


 English  and  CAPITAL IZATION 
 � � �  � � � 
 show _ tokens  False  None  elif  ==  >=  else :  two  tabs :"      "  Three  tabs :  "         " 
 12 . 0 * 50 = 600 
 

**StarCoder2.** Tokenizador especializado em código-fonte. Note como espaços consecutivos e tabulações (comuns em codigo indentado) tendem a virar tokens próprios, algo pouco útil para um tokenizador de texto natural mas essencial para representar programas de forma eficiente.


In [15]:
# You need to request access before being able to use this tokenizer
show_tokens(text, "bigcode/starcoder2-15b")

config.json:   0%|          | 0.00/803 [00:00<?, ?B/s]

[transformers] Model config: bos_token_id must be `None` or an integer within the vocabulary (between 0 and 49151), got 50256. This may result in unexpected behavior.
[transformers] Model config: eos_token_id must be `None` or an integer within the vocabulary (between 0 and 49151), got 50256. This may result in unexpected behavior.


tokenizer_config.json:   0%|          | 0.00/7.88k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/777k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/442k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/958 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.06M [00:00<?, ?B/s]

[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer GPT2Tokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.



 English  and  CAPITAL IZATION 
 � � �   � � 
 show _ tokens  False  None  elif  ==  >=  else :  two  tabs :"      "  Three  tabs :  "         " 
 1 2 . 0 * 5 0 = 6 0 0 
 

**Galactica.** Tokenizador treinado majoritariamente em literatura científica, o que pode gerar tokens próprios para notação numerica e símbolos matemáticos.


In [16]:
show_tokens(text, "facebook/galactica-1.3b")

config.json:   0%|          | 0.00/789 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/166 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.14M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/3.00 [00:00<?, ?B/s]


 English  and  CAP ITAL IZATION 
 � � � �  � � � 
 show _ tokens  False  None  elif   ==   > =  else :  two  t abs : "      "  Three  t abs :   "         " 
 1 2 . 0 * 5 0 = 6 0 0 
 

**Phi-3-mini.** O mesmo tokenizador do modelo que carregamos no início do notebook para geração de texto, permitindo comparar diretamente com todos os outros tokenizadores acima.


In [17]:
show_tokens(text, "microsoft/Phi-3-mini-4k-instruct")

 
 English and C AP IT AL IZ ATION 
 � � � �  � � � 
 show _ to kens False None elif == >= else : two tabs :"    " Three tabs : "       " 
 1 2 . 0 * 5 0 = 6 0 0 
 

# Embeddings contextualizados de palavras (com um modelo tipo BERT)

Até aqui trabalhamos apenas com tokens e IDs. Agora avançamos para a etapa seguinte do pipeline mostrado em aula: tokens são convertidos em **vetores de embedding**, que depois passam pelas camadas do modelo de linguagem e saem como **embeddings contextuais**, ou seja, vetores que incorporam o contexto da frase inteira, e não apenas o significado isolado da palavra. E a diferença entre a tabela estática de embeddings do tokenizador e a representação final, dinâmica, que o modelo produz após processar a sequência.


Carregamos o tokenizador e o modelo **DeBERTa** (uma variante da [família BERT](https://www.deeplearningbook.com.br/o-que-e-bert-bidirectional-encoder-representations-from-transformers/)), tokenizamos a frase "Hello world" e fazemos um *forward pass* pelo modelo. A saída `output` contém, para cada token de entrada, um vetor contextualizado: e a materialização prática do "Embeddings" -> "Language model" -> "Contextual token embedding vectors" do diagrama visto em aula.


In [18]:
from transformers import AutoModel, AutoTokenizer

# Load a tokenizer
tokenizer = AutoTokenizer.from_pretrained("microsoft/deberta-base")

# Load a language model
model = AutoModel.from_pretrained("microsoft/deberta-v3-xsmall")

# Tokenize the sentence
tokens = tokenizer('Hello world', return_tensors='pt')

# Process the tokens
output = model(**tokens)[0]

config.json:   0%|          | 0.00/474 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/578 [00:00<?, ?B/s]

pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  241MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

[transformers] DebertaV2Model LOAD REPORT from: microsoft/deberta-v3-xsmall
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
deberta.embeddings.word_embeddings._weight | UNEXPECTED |  | 
lm_predictions.lm_head.dense.bias          | UNEXPECTED |  | 
lm_predictions.lm_head.dense.weight        | UNEXPECTED |  | 
lm_predictions.lm_head.LayerNorm.bias      | UNEXPECTED |  | 
mask_predictions.dense.weight              | UNEXPECTED |  | 
mask_predictions.classifier.bias           | UNEXPECTED |  | 
mask_predictions.classifier.weight         | UNEXPECTED |  | 
lm_predictions.lm_head.LayerNorm.weight    | UNEXPECTED |  | 
mask_predictions.LayerNorm.bias            | UNEXPECTED |  | 
mask_predictions.LayerNorm.weight          | UNEXPECTED |  | 
lm_predictions.lm_head.bias                | UNEXPECTED |  | 
mask_predictions.dense.bias                | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from d

O formato (`shape`) do tensor de saída segue o padrão `(número de sentencas, número de tokens, dimensão do embedding)`. A terceira dimensão e o tamanho do vetor que representa cada token, análogo as "~300 dimensões" citadas no slide "Embeddings" para comprimir um vocabulário que, em One-Hot, teria dezenas de milhares de dimensões esparsas.


In [19]:
output.shape

torch.Size([1, 4, 384])

Aqui decodificamos os IDs de entrada para conferir quais tokens o DeBERTa efetivamente gerou para "Hello world". Modelos da família BERT costumam adicionar tokens especiais de início/fim de sequência (por exemplo `[CLS]` e `[SEP]`), que tambem recebem seus próprios embeddings.


In [20]:
for token in tokens['input_ids'][0]:
    print(tokenizer.decode(token))

[CLS]
Hello
 world
[SEP]


`output` mostra os valores numéricos brutos dos embeddings contextualizados, um vetor por token. É o análogo prático do vetor `Batata = [2.34, 4.75, 17.01, -0.223]` mostrado no slide "Embeddings": cada linha aqui e um vetor denso e contínuo, só que agora calculado por um modelo real e já levando em conta
o contexto da frase, não apenas a identidade da palavra isolada.


In [21]:
output

tensor([[[-3.4805,  0.0862, -0.1818,  ..., -0.0610, -0.3909,  0.3022],
         [ 0.1885,  0.3201, -0.2313,  ...,  0.3721,  0.2471,  0.8057],
         [ 0.2089,  0.5010, -0.0495,  ...,  1.2197, -0.2277,  0.8574],
         [-3.4277,  0.0635, -0.1426,  ...,  0.0658, -0.4358,  0.3826]]],
       dtype=torch.float16, grad_fn=<NativeLayerNormBackward0>)

# Embeddings de texto (para frases e documentos inteiros)

Ate agora cada **token** tinha seu próprio vetor. Muitas aplicações (busca semântica, recomendação, agrupamento de documentos) precisam, em vez disso, de **um único vetor por frase ou documento**. A biblioteca `sentence-transformers` resolve isso combinando (fazendo *pooling*) os embeddings contextualizados de todos os tokens de uma frase em um vetor só, de tamanho fixo, que resume o
significado da frase inteira.


Carregamos um modelo de sentence embeddings (`all-mpnet-base-v2`) e convertemos a frase "Best movie ever!" diretamente em um único vetor numérico. Esse vetor pode então ser comparado, via similaridade de cosseno (a mesma métrica do slide "Embeddings de Personalidade: Como voce e?"), com o vetor de outras frases para medir o quanto elas são semanticamente próximas.


In [22]:
from sentence_transformers import SentenceTransformer

# Load model
model = SentenceTransformer('sentence-transformers/all-mpnet-base-v2')

# Convert text to text embeddings
vector = model.encode("Best movie ever!")

model.safetensors: reconstructing file:   0%|          |  0.00B /  241MB            

model.safetensors: downloading bytes:           |  0.00B            

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/11.6k [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/363 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

O `shape` aqui e apenas uma dimensão: o tamanho do vetor que representa a frase inteira (não há mais uma dimensão de "número de tokens", pois todos os tokens já foram combinados em um único vetor).


In [23]:
vector.shape

(768,)

In [37]:
vector

array([-2.02203933e-02,  4.57696877e-02, -1.26637146e-02, -3.37991631e-03,
       -2.54910300e-03, -3.31096235e-03, -4.58366945e-02,  2.32390799e-02,
       -3.12585123e-02, -3.15588824e-02, -2.19783355e-02,  3.67821753e-03,
        3.39105981e-03, -1.88130587e-02, -2.65821870e-02, -1.45892315e-02,
        3.20066400e-02, -3.66493082e-03,  1.75410733e-02,  5.43209612e-02,
       -3.96661647e-02,  9.93637647e-03, -3.27205285e-02, -1.62909944e-02,
        6.36525359e-03, -1.24459760e-02,  1.04737049e-02,  3.08674313e-02,
        7.47453189e-03, -3.86319868e-02, -6.19924963e-02, -3.25308628e-02,
       -1.40268244e-02,  5.16586639e-02,  1.69651707e-06, -1.74092929e-04,
       -1.27881172e-03, -4.15052772e-02,  5.00347884e-03,  4.58600894e-02,
       -2.85337027e-02,  5.02370670e-02, -1.75183658e-02, -3.98830482e-04,
        1.63311344e-02,  5.89936748e-02, -2.21730098e-02, -4.00975086e-02,
       -6.60439581e-03, -2.97632981e-02, -3.18221636e-02, -1.77945811e-02,
       -1.54363075e-02, -

# Embeddings de palavras alem dos LLMs

Nem todo embedding vem de um LLM baseado em Transformer. O **Word2Vec** (e sua variante treinada em dados da Wikipedia, o **GloVe**, que usaremos aqui) e o método clássico apresentado nos slides "Word2Vec": os embeddings são aprendidos a partir de uma tarefa de classificação simples, prever se duas palavras são vizinhas dentro de uma janela deslizante de texto (*skip-gram*), com exemplos
negativos de palavras que normalmente não aparecem juntas (*negative  sampling*). Diferente dos embeddings contextualizados vistos acima, aqui cada palavra tem **um unico vetor fixo**, independente do contexto em que aparece.


Baixamos, via `gensim`, um conjunto de embeddings GloVe ja treinados na Wikipedia (vetores de 50 dimensões). Note que isso é apenas *carregar* pesos ja prontos, não estamos treinando nada nesta célula.


In [38]:
import gensim.downloader as api

# Download embeddings (66MB, glove, trained on wikipedia, vector size: 50)
# Other options include "word2vec-google-news-300"
# More options at https://github.com/RaRe-Technologies/gensim-data
model = api.load("glove-wiki-gigaword-50")

Aqui reproduzimos, com dados reais, a famosa analogia vetorial mostrada no slide "Analogias": `king - man + woman ~= queen`. O método `most_similar` calcula a similaridade de cosseno entre o vetor de `king` e todos os demais vetores do vocabulário, retornando os mais próximos. Se a geometria dos
embeddings capturou bem a relação semantica de gênero e realeza, `queen` deve aparecer entre os primeiros resultados, exatamente como no slide.


In [25]:
model.most_similar([model['king']], topn=11)

[('king', 1.0000001192092896),
 ('prince', 0.8236179351806641),
 ('queen', 0.7839043140411377),
 ('ii', 0.7746230363845825),
 ('emperor', 0.7736247777938843),
 ('son', 0.766719400882721),
 ('uncle', 0.7627150416374207),
 ('kingdom', 0.7542161345481873),
 ('throne', 0.7539914846420288),
 ('brother', 0.7492411136627197),
 ('ruler', 0.7434253692626953)]

In [39]:
model.most_similar(positive=["king", "woman"], negative = ["man"])

[('queen', 0.8523604273796082),
 ('throne', 0.7664334177970886),
 ('prince', 0.7592144012451172),
 ('daughter', 0.7473883628845215),
 ('elizabeth', 0.7460219860076904),
 ('princess', 0.7424570322036743),
 ('kingdom', 0.7337412238121033),
 ('monarch', 0.721449077129364),
 ('eldest', 0.7184861898422241),
 ('widow', 0.7099431157112122)]

In [43]:
model.most_similar(positive=["dog"])

[('cat', 0.9218004941940308),
 ('dogs', 0.8513158559799194),
 ('horse', 0.7907583713531494),
 ('puppy', 0.7754920721054077),
 ('pet', 0.7724708318710327),
 ('rabbit', 0.7720814347267151),
 ('pig', 0.7490062117576599),
 ('snake', 0.7399188876152039),
 ('baby', 0.7395570278167725),
 ('bite', 0.7387937307357788)]

In [44]:
model.most_similar(positive=["school"])

[('college', 0.9344995617866516),
 ('schools', 0.868353009223938),
 ('campus', 0.8472232222557068),
 ('graduate', 0.8460071682929993),
 ('elementary', 0.8369437456130981),
 ('students', 0.8265615105628967),
 ('university', 0.8261534571647644),
 ('student', 0.8239262104034424),
 ('attended', 0.8064848184585571),
 ('graduating', 0.8018305897712708)]

In [45]:
model.most_similar(positive=["july"])

[('june', 0.9965305328369141),
 ('april', 0.9958918690681458),
 ('march', 0.9862644076347351),
 ('december', 0.9852781891822815),
 ('november', 0.9847837090492249),
 ('september', 0.9842234253883362),
 ('october', 0.9841668009757996),
 ('february', 0.9822770357131958),
 ('august', 0.9822704195976257),
 ('january', 0.9798345565795898)]

In [ ]:
# Tente algum...

# Recomendando músicas com embeddings

Esta última seção generaliza a ideia central da aula: **o algoritmo Word2Vec  não sabe nada sobre "palavras"**, ele apenas aprende a partir de sequências de símbolos que aparecem juntos com frequência. Se trocarmos "frases de texto" por "playlists" e "palavras" por "músicas", a mesma lógica de treinamento skip-gram com negative sampling, slide "Word2Vec") pode ser usada para aprender embeddings de músicas a partir de listas de reprodução, e depois recomendar músicas semelhantes usando similaridade de cosseno, exatamente como fizemos com `king` e `queen` acima.


Baixamos um dataset público de playlists (cada linha e uma playlist, representada como uma lista de IDs de musica) e a tabela de metadados das músicas (título e artista por ID). Aqui, cada **playlist** faz o papel de uma "frase" e cada **musica** faz o papel de uma "palavra" dentro dessa frase.


In [47]:
import pandas as pd
from urllib import request

# Get the playlist dataset file
data = request.urlopen('https://storage.googleapis.com/maps-premium/dataset/yes_complete/train.txt')

# Parse the playlist dataset file. Skip the first two lines as
# they only contain metadata
lines = data.read().decode("utf-8").split('\n')[2:]

# Remove playlists with only one song
playlists = [s.rstrip().split() for s in lines if len(s.split()) > 1]

# Load song metadata
songs_file = request.urlopen('https://storage.googleapis.com/maps-premium/dataset/yes_complete/song_hash.txt')
songs_file = songs_file.read().decode("utf-8").split('\n')
songs = [s.rstrip().split('\t') for s in songs_file]
songs_df = pd.DataFrame(data=songs, columns = ['id', 'title', 'artist'])
songs_df = songs_df.set_index('id')
songs_df.to_csv("song_hash.csv")
songs_df

,title,artist
id,,
0,Gucci Time (w\/ Swizz Beatz),Gucci Mane
1,Aston Martin Music (w\/ Drake & Chrisette Mich...,Rick Ross
2,Get Back Up (w\/ Chris Brown),T.I.
3,Hot Toddy (w\/ Jay-Z & Ester Dean),Usher
4,Whip My Hair,Willow
...,...,...
75258,USA Today,Alan Jackson
75259,Superstar,Raul Malo
75260,Romancin' The Blues,Giacomo Gates


Conferimos o formato dos dados brutos: cada playlist e simplesmente uma sequência de IDs de musica, na ordem em que aparecem na lista de reprodução.


In [27]:
print( 'Playlist #1:\n ', playlists[0], '\n')
print( 'Playlist #2:\n ', playlists[1])

Playlist #1:
  ['0', '1', '2', '3', '4', '5', '6', '7', '8', '9', '10', '11', '12', '13', '14', '15', '16', '17', '18', '19', '20', '21', '22', '23', '24', '25', '26', '27', '28', '29', '30', '31', '32', '33', '34', '35', '36', '37', '38', '39', '40', '41', '2', '42', '43', '44', '45', '46', '47', '48', '20', '49', '8', '50', '51', '52', '53', '54', '55', '56', '57', '25', '58', '59', '60', '61', '62', '3', '63', '64', '65', '66', '46', '47', '67', '2', '48', '68', '69', '70', '57', '50', '71', '72', '53', '73', '25', '74', '59', '20', '46', '75', '76', '77', '59', '20', '43'] 

Playlist #2:
  ['78', '79', '80', '3', '62', '81', '14', '82', '48', '83', '84', '17', '85', '86', '87', '88', '74', '89', '90', '91', '4', '73', '62', '92', '17', '53', '59', '93', '94', '51', '50', '27', '95', '48', '96', '97', '98', '99', '100', '57', '101', '102', '25', '103', '3', '104', '105', '106', '107', '47', '108', '109', '110', '111', '112', '113', '25', '63', '62', '114', '115', '84', '116', '117',

Aqui treinamos nosso próprio modelo **Word2Vec** do zero, tratando cada playlist como uma sequência de tokens (as músicas) e cada música como um token. Os parâmetros `vector_size`, `window` e `negative` correspondem exatamente aos conceitos do slide "Word2Vec": tamanho do vetor de embedding, tamanho da
janela de contexto (quantas músicas vizinhas contam como "positivas") e número de amostras negativas usadas para evitar que o modelo aprenda a "trapacear" prevendo sempre 1.


In [28]:
from gensim.models import Word2Vec

# Train our Word2Vec model
model = Word2Vec(
    playlists, vector_size=32, window=20, negative=50, min_count=1, workers=4
)

Com o modelo treinado, pedimos as músicas mais similares (por similaridade de cosseno no espaço de embeddings aprendido) a música de ID 2172. O resultado são os IDs das músicas vizinhas nesse espaço vetorial, o equivalente musical do `model.most_similar(positive=["king","woman"], negative=["man"])` visto na seção anterior.


In [29]:
song_id = 2172

# Ask the model for songs similar to song #2172
model.wv.most_similar(positive=str(song_id))

[('2976', 0.9980632066726685),
 ('3167', 0.995846688747406),
 ('3114', 0.9954792261123657),
 ('6706', 0.9949949383735657),
 ('5493', 0.9949274659156799),
 ('2640', 0.9943389296531677),
 ('2849', 0.9940239191055298),
 ('5549', 0.9932549595832825),
 ('2886', 0.9931840300559998),
 ('2881', 0.992957592010498)]

Consultamos a tabela de metadados para descobrir título e artista da música de ID 2172, que serviu de referencia na busca por similares.


In [48]:
songs_df.iloc[2172]

,2172
title,Fade To Black
artist,Metallica


Esta função encapsula o fluxo completo de recomendação: busca as músicas mais próximas no espaço de embeddings e traduz os IDs retornados em título e artista, usando a tabela `songs_df`. E a aplicação final e mais concreta do princípio de Firth citado na aula ("uma palavra e conhecida pela companhia que
mantem"): aqui, "uma música e conhecida pelas playlists em que aparece".


In [31]:
import numpy as np

def print_recommendations(song_id):
    similar_songs = np.array(
        model.wv.most_similar(positive=str(song_id),topn=5)
    )[:,0]
    return  songs_df.iloc[similar_songs]

# Extract recommendations
print_recommendations(2172)

,title,artist
id,,
2976,I Don't Know,Ozzy Osbourne
3167,Unchained,Van Halen
3114,And The Cradle Will Rock,Van Halen
6706,Break On Through (To The Other Side),The Doors
5493,Shot In The Dark,Ozzy Osbourne


In [32]:
print_recommendations(2172)

,title,artist
id,,
2976,I Don't Know,Ozzy Osbourne
3167,Unchained,Van Halen
3114,And The Cradle Will Rock,Van Halen
6706,Break On Through (To The Other Side),The Doors
5493,Shot In The Dark,Ozzy Osbourne


In [49]:
songs_df.iloc[842]

,842
title,California Love (w\/ Dr. Dre & Roger Troutman)
artist,2Pac


In [33]:
print_recommendations(842)

,title,artist
id,,
12205,Give It Up To Me,Sean Paul
1560,In Da Club,50 Cent
413,If I Ruled The World (Imagine That) (w\/ Laury...,Nas
27078,Out Of My Head (w\/ Trey Songz),Lupe Fiasco
886,Heartless,Kanye West


## Referências

- Aula "Tokens e Embeddings", Topicos Especiais em Inteligencia Artificial, Prof. Rodrigo Lira, IFPE
  Campus Paulista.
- Alammar, J.; Grootendorst, M. *Hands-On Large Language Models*, Capitulo 2 (Tokens and Token
  Embeddings). O'Reilly, 2024.
- Alammar, J. *The Illustrated Word2vec*. Disponivel em: https://jalammar.github.io/illustrated-word2vec/
- Hugging Face. *Tokenizers documentation*. Disponivel em: https://huggingface.co/docs/tokenizers/index
- Cornell University. *Learning Latent Vector Spaces for Product Search / Playlist dataset*.
  Disponivel em: https://www.cs.cornell.edu/~shuochen/lme/data_page.html
